# CIE 2 – Activity 2: Simple House Price Prediction

**Course:** Machine Learning (ADPC509)  
**Unit:** II – Supervised Learning: Regression  
**Marks:** 5

## 1. Dataset selection and source
This notebook uses the **California Housing dataset** provided by scikit-learn. It contains California district-level housing information and a continuous target, median house value.

- Dataset documentation: https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset
- Loader: `sklearn.datasets.fetch_california_housing`
- Target units: hundreds of thousands of US dollars (the dataset target is expressed in units of $100,000).
- Model scope: a single-feature Linear Regression model using `MedInc` (median income) to predict median house value.

The dataset is downloaded by scikit-learn when the notebook is run in Google Colab or another environment with internet access.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Download and load the dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())
display(df.tail())
print("\nData types:")
display(df.dtypes.to_frame("Data type"))

## 2. Inspect and prepare the data

In [ ]:
print("Summary statistics:")
display(df.describe().T)

print("Missing values:")
display(df.isna().sum().to_frame("Missing values"))

print("Duplicate rows:", df.duplicated().sum())

# Remove exact duplicate records only if present.
df = df.drop_duplicates().copy()

# Remove rows with missing values in the selected feature or target, if any.
df = df.dropna(subset=["MedInc", "MedHouseVal"]).copy()

print("Shape after basic cleaning:", df.shape)

## 3. Select input feature (X) and target (y)

- **Input X:** `MedInc` – median income in the district.
- **Target y:** `MedHouseVal` – median house value.

Median income is selected because household income can be related to the value of houses in an area. This relationship does not prove that income causes house prices to change.

In [ ]:
X = df[["MedInc"]]
y = df["MedHouseVal"]

print("Input feature:", X.columns.tolist())
print("Target:", y.name)
display(pd.concat([X, y], axis=1).head())

## 4. Visualize the relationship

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X["MedInc"], y, alpha=0.25)
plt.title("Median Income vs Median House Value")
plt.xlabel("Median Income (dataset units)")
plt.ylabel("Median House Value (units of $100,000)")
plt.tight_layout()
plt.show()

print("Observation: inspect the scatter plot to determine the direction, strength and possible non-linearity of the relationship.")
print("A visible relationship does not by itself prove causation.")

## 5. Split the data into training and testing sets

An 80:20 split is used. The training set is used to learn model parameters, while the test set is kept separate for evaluation on unseen observations.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

## 6. Train Linear Regression

For a single-feature model, the equation is:

**Predicted house value = intercept + coefficient × median income**

The coefficient represents the expected change in predicted median house value for a one-unit increase in median income. The intercept is the model's predicted value when median income is zero.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

coefficient = model.coef_[0]
intercept = model.intercept_

print("Coefficient:", coefficient)
print("Intercept:", intercept)
print(f"Model equation: y = {intercept:.4f} + ({coefficient:.4f} × MedInc)")

## 7. Generate test predictions

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual House Value": y_test.values,
    "Predicted House Value": y_pred
})
display(comparison.head(10))

## 8. Evaluate the model

- **MAE:** average absolute prediction error.
- **MSE:** average squared prediction error.
- **RMSE:** square root of MSE, expressed in the target's units.
- **R²:** compares the model with a mean-based baseline; it can be negative on test data.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r2]
})
display(metrics)

print(f"MAE: {mae:.4f} (in units of $100,000)")
print(f"MSE: {mse:.4f} (squared target units)")
print(f"RMSE: {rmse:.4f} (in units of $100,000)")
print(f"R²: {r2:.4f}")

## 9. Visualize actual versus predicted values

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.25)
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title("Actual vs Predicted House Values")
plt.xlabel("Actual House Value (units of $100,000)")
plt.ylabel("Predicted House Value (units of $100,000)")
plt.tight_layout()
plt.show()

print("Points closer to the dashed reference line represent more accurate predictions.")

## 10. Predict a new example

In [ ]:
# Example: median income value of 5.0 dataset units.
new_example = pd.DataFrame({"MedInc": [5.0]})
new_prediction = model.predict(new_example)[0]

print("New input – MedInc:", new_example.loc[0, "MedInc"])
print(f"Predicted median house value: {new_prediction:.4f} units of $100,000")
print(f"Approximate value in US dollars: ${new_prediction * 100000:,.2f}")

## Conclusion

A single-feature Linear Regression model was developed using median income to predict median house value in the California Housing dataset. The data was inspected, cleaned where necessary, split into training and testing sets, and used to train the model. The coefficient and intercept define the fitted regression equation. MAE, MSE, RMSE and R² are calculated on the test set, and actual-versus-predicted values are visualized.

**Limitations:** house values depend on several factors, while this model uses only median income. The relationship may not be perfectly linear, and a single-feature model cannot capture all local housing differences. The metric values and observations should be taken from the executed notebook output.

## GitHub publication checklist

Create a separate **public** repository, for example `ML-CIE2-Activity2-YourRollNumber`, and upload:
- This notebook (`Activity_2_House_Price_Prediction.ipynb`)
- A `README.md` describing the dataset, model, metrics and results after execution

Add this acknowledgement to the README:

> This project was completed as part of the Machine Learning CIE 2 activity. Faculty reference and Machine Learning learning resources: https://github.com/Princy2129

Faculty profile: https://github.com/Princy2129